IMPORTS

WANG'S EXAMPLES

In [ ]:
import os
import os.path as op

import openneuro
from mne.datasets import sample

from mne_bids import (
    BIDSPath,
    find_matching_paths,
    get_entity_vals,
    make_report,
    print_dir_tree,
    read_raw_bids,
)
import os
from os import path
import openneuro
from mne.datasets import sample
import numpy as np
import matplotlib.pyplot as plt
import mne
from mne.viz import plot_topomap

from naplib.io import load_bids

In [ ]:
import mne  # Import MNE

# Download the sample dataset if it is not already downloaded
data_path = mne.datasets.sample.data_path(download=True)

print("MNE sample dataset path:", data_path)

In [ ]:
# Define the dataset ID
dataset = "ds002778"

# Define the root directory for storing the BIDS dataset
bids_root = op.join(op.dirname(sample.data_path()), dataset)

# Create the directory if it doesn't exist
if not op.isdir(bids_root):
    os.makedirs(bids_root)

In [ ]:
from mne_bids import print_dir_tree

print_dir_tree(bids_root, max_depth=4)


In [ ]:
#dataset = 'ds002778'
#subject = 'pd6'

#bids_root = path.join(path.dirname(sample.data_path()), dataset)
#print(bids_root)
#if not path.isdir(bids_root):
    #os.makedirs(bids_root)

#openneuro.download(dataset=dataset, target_dir=bids_root,
                  # include=[f'sub-{subject}'])

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "pd6"
session = "off"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


# Print EEG data info
print(raw.info)
raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='off', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded


In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)



In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc1"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


# Print EEG data info
print(raw.info)
raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded


In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc1 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc7"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


# Print EEG data info
#print(raw.info)
#raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc7 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc8"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


# Print EEG data info
#print(raw.info)
#raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc8 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc10"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

# Print EEG data info
#print(raw.info)
#raw.plot()


In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)


In [ ]:

# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])


In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc10 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc18"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

# Print EEG data info
#print(raw.info)
#raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc18 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc20"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

# Print EEG data info
#print(raw.info)
#raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('Hc20 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc2"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

# Print EEG data info
#print(raw.info)
#raw.plot()

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:

print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded

In [ ]:
# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
# Extract EEG Features
# Computes Alpha/Theta Ratio per electrode.

import numpy as np
import mne

# Compute Power Spectral Density (PSD) using Welch's method
psd = raw.compute_psd(method="welch", fmin=1, fmax=40, n_fft=1024)
psds, freqs = psd.get_data(return_freqs=True)  # Extract PSD values and frequencies

# Define Alpha (8-12 Hz) and Theta (4-7 Hz) bands
alpha_idx = np.logical_and(freqs >= 8, freqs <= 12)
theta_idx = np.logical_and(freqs >= 4, freqs <= 7)

# Compute Alpha/Theta Ratio
alpha_power = psds[:, alpha_idx].mean(axis=1)
theta_power = psds[:, theta_idx].mean(axis=1)
alpha_theta_ratio = np.log(alpha_power / theta_power)  # Log transform

print("Alpha/Theta Ratio per Channel:", alpha_theta_ratio)

In [ ]:
# Visualize the Alpha/Theta Ratio as a Topomap
# Generates a topographic map of Alpha/Theta ratio.

import matplotlib.pyplot as plt
from mne.viz import plot_topomap

fig, ax = plt.subplots()
ax.set_title('hc2 Alpha/Theta Ratio')
plot_topomap(alpha_theta_ratio, raw.info, axes=ax)
plt.show()

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc4"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)


In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:

print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:

# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc4"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc21"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc24"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc25"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc29"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc30"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc31"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc32"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts

In [ ]:
from mne_bids import BIDSPath, read_raw_bids

# Example: Load EEG data for one subject (modify subject/session as needed)
subject = "hc33"
session = "hc"  # or "on" for ON medication state
datatype = "eeg"
task = "rest"
suffix = "eeg"

bids_path = BIDSPath(root=bids_root, subject=subject, session=session, datatype=datatype, task=task, suffix=suffix)
raw = read_raw_bids(bids_path)

In [ ]:
# We are only interested in the 32-channel EEG data as the responses, so select those channels
resp_channels = ['Fp1','AF3','F7','F3','FC1','FC5','T7','C3','CP1','CP5','P7',
                 'P3','Pz','PO3','O1','Oz','O2','PO4','P4','P8','CP6','CP2',
                 'C4','T8','FC6','FC2','F4','F8','AF4','Fp2','Fz','Cz']

data = load_bids(root=bids_root, subject=subject, datatype='eeg', task='rest', suffix='eeg', session='hc', resp_channels=resp_channels)

In [ ]:
# Apply Notch Filter (Remove Line Noise at 50/60 Hz)
# Removes power line interference.
# Explicitly load data into memory
raw.load_data()  # This replaces preload=True

# Print EEG data info
#print(raw.info)


raw.notch_filter(freqs=[50, 60])

In [ ]:
print(bids_path)  # Check if path is correct
print(raw)  # Check if data is successfully loaded



# Keeps 1-40 Hz signals (filters out unwanted frequencies).
raw.filter(l_freq=1.0, h_freq=40.0)

In [ ]:
# Compute and plot PSD (Power Spectral Density)
psd = raw.compute_psd()
psd.plot(amplitude=False)  # Corrected way to plot PSD

# Handle the missing channels by setting them as EOG or other types
mapping = {
    "EXG1": "eog", "EXG2": "eog", "EXG3": "eog", "EXG4": "eog",
    "EXG5": "eog", "EXG6": "eog", "EXG7": "eog", "EXG8": "eog"
}
raw.set_channel_types(mapping)  # Set non-EEG channels as EOG

# Assign the standard 10-20 montage correctly
raw.set_montage("standard_1020", on_missing="ignore")  # Ignore missing channel positions

In [ ]:
# Detect and Remove Eye Blinks & Movement Artifacts (ICA)
# 
from mne.preprocessing import ICA

ica = ICA(n_components=20, random_state=97)
ica.fit(raw)
ica.plot_components()  # Identify eye/motion artifacts
raw = ica.apply(raw)  # Remove artifacts